In [37]:
import numpy as np

class Conv:

    def __init__(self, num_filters, kernal_size, learn_rate):
        self.num_filters = num_filters
        self.lr = learn_rate
        self.kernal_size = kernal_size

        self.filters = np.random.randn(num_filters, self.kernal_size[0], self.kernal_size[1])
        self.filters = self.filters / np.amax(self.filters, axis=(0,1,2))

    def iterate_regions(self, image):
        # valid padding
        h, w = image.shape

        for i in range(h-self.kernal_size[0]+1):
            for j in range(w-self.kernal_size[1]+1):
                im_region = image[i:(i+self.kernal_size[0]), j:(j+self.kernal_size[1])]
                yield im_region, i, j

    def forward(self, input):
        h, w = input.shape
        output = np.zeros((h-self.kernal_size[0]+1, w-self.kernal_size[1]+1, self.num_filters))
        self.last_input = input

        for im_region, i, j in self.iterate_regions(input):
            output[i,j] = np.sum(im_region * self.filters, axis=(1, 2))

        return output
    
    def backprop(self, d_L_d_out):
        d_L_d_filters = np.zeros(self.filters.shape)

        for im_region, i, j in self.iterate_regions(self.last_input):
            for f in range(self.num_filters):
                d_L_d_filters[f] += d_L_d_out[i, j, f] * im_region

        self.filters -= self.lr * d_L_d_filters
    
        return None
    

In [2]:
import os
from PIL import Image

PATH = "./CN_MNIST/data/data"
img_num = 4500
files = os.listdir(PATH)[:img_num]
data = []
labels = []
kinds = 15

for i, file in enumerate(files):
    img = Image.open(PATH + '/' + file)
    data.append(np.array(img))
    _, _, _, label = file.split("_")
    label, _ = label.split(".")
    labels.append(int(label) - 1)

data = np.array(data)



In [3]:
class MaxPool2:

    def iterate_regions(self, image):
        h, w, _ = image.shape
        new_h = h // 2
        new_w = w // 2

        for i in range(new_h):
            for j in range(new_w):
                im_region = image[(i*2):(i*2+2), (j*2):(j*2+2)]
                yield im_region, i, j

    def forward(self, input):
        h, w, num_filters = input.shape
        output = np.zeros((h // 2, w // 2, num_filters))
        self.last_input = input

        for im_region, i, j in self.iterate_regions(input):
            output[i,j] = np.amax(im_region, axis=(0, 1))

        return output
    
    def backprop(self, d_L_d_out):
        d_L_d_input = np.zeros(self.last_input.shape)

        for im_region, i, j in self.iterate_regions(self.last_input):
            h, w, f = im_region.shape
            amax = np.amax(self.last_input, axis=(0, 1))

            for i2 in range(h):
                for j2 in range(w):
                    for k2 in range(f):
                        if im_region[i2, j2, k2] == amax[k2]:
                            d_L_d_input[i*2 + i2, j*2 + j2, k2] = d_L_d_out[i, j, k2]

        return d_L_d_input


In [4]:
class Softmax:

    def __init__(self, input_len, nodes, learn_rate):
        self.weights = np.random.randn(input_len, nodes)
        self.weights = self.weights / np.amax(self.weights, axis=(0,1))
        self.biases = np.zeros(nodes)
        self.lr = learn_rate

    def forward(self, input):
        self.last_input_shape = input.shape

        input = input.flatten()
        self.last_input = input

        input_len, nodes = self.weights.shape

        totals = np.dot(input, self.weights) + self.biases
        self.last_totals = totals

        exp = np.exp(totals)
        return exp / np.sum(exp, axis=0)
    
    def backprop(self, d_L_d_out):
        for i, gradient in enumerate(d_L_d_out):
            if gradient == 0:
                continue

            exp_t = np.exp(self.last_totals)

            S = np.sum(exp_t)

            d_out_d_t = - exp_t * exp_t[i] / (S ** 2)
            d_out_d_t[i] = exp_t[i] * (S - exp_t[i]) / (S ** 2)

            d_t_d_w = self.last_input
            d_t_d_inputs = self.weights 
            d_L_d_t = gradient * d_out_d_t

            d_L_d_w = d_t_d_w[np.newaxis].T @ d_L_d_t[np.newaxis]
            d_L_d_b = d_L_d_t
            d_L_d_inputs = d_t_d_inputs @ d_L_d_t

            self.weights -= self.lr * d_L_d_w
            self.biases -= self.lr * d_L_d_b
            return d_L_d_inputs.reshape(self.last_input_shape)




In [ ]:


conv = Conv(8, (2,2), learn_rate=0.001) # 64x64 -> 62x62x8
pool = MaxPool2()                       # 62x62x8 -> 31x31x8
fc = Softmax(31 * 31 * 8, kinds, learn_rate=0.001)

def forward(image, label):
    out = conv.forward((image / 255) - 0.5)
    out = pool.forward(out)
    out = fc.forward(out)

    loss = -np.log(out[label])
    acc = 1 if np.argmax(out) == label else 0

    return out, loss, acc

def train(img, label):
    out, loss, acc = forward(img, label)

    
    gradient = np.zeros(kinds)
    gradient[label] = - 1 / out[label]

    gradient = fc.backprop(gradient)
    gradient = pool.backprop(gradient)
    gradient = conv.backprop(gradient)

    return loss, acc




In [46]:
from sklearn.model_selection import train_test_split

train_images, test_images, train_labels, test_labels = train_test_split(data, labels, test_size=0.1)

In [47]:
epoches = 20

for epoch in range(epoches):
    print(f"\nEpoch {epoch + 1}:")

    permulation = np.random.permutation(len(train_images))
    train_images = train_images[permulation]
    train_labels = np.array(train_labels)[permulation]
    
    loss = 0
    num_correct = 0

    for i, (img, label) in enumerate(zip(train_images, train_labels)):
        l, acc = train(img, label)
        loss += l
        num_correct += acc

        if i > 0 and i % 10 == 9:
            print("\r[Step %d] Average Loss: %.3f | Accuracy: %.2f" %
                  (i + 1, loss / (i + 1), num_correct / (i + 1)), end='', flush=True
                  )


print("\nTrain finished!")

print("Testing the CNN:")
loss = 0
num_correct = 0
for img, label in zip(test_images, test_labels):
    _, l, acc = forward(img, label)

    loss += l
    num_correct += acc

num_tests = len(test_images)
print("Test Loss:", loss / num_tests)
print("Test Accuracy:", num_correct / num_tests)
    


Epoch 1:
[Step 4050] Average Loss: 3.373 | Accuracy: 0.10
Epoch 2:
[Step 4050] Average Loss: 2.991 | Accuracy: 0.14
Epoch 3:
[Step 4050] Average Loss: 2.743 | Accuracy: 0.18
Epoch 4:
[Step 4050] Average Loss: 2.552 | Accuracy: 0.23
Epoch 5:
[Step 4050] Average Loss: 2.404 | Accuracy: 0.26
Epoch 6:
[Step 4050] Average Loss: 2.291 | Accuracy: 0.29
Epoch 7:
[Step 4050] Average Loss: 2.191 | Accuracy: 0.31
Epoch 8:
[Step 4050] Average Loss: 2.101 | Accuracy: 0.35
Epoch 9:
[Step 4050] Average Loss: 2.027 | Accuracy: 0.38
Epoch 10:
[Step 4050] Average Loss: 1.970 | Accuracy: 0.40
Epoch 11:
[Step 4050] Average Loss: 1.917 | Accuracy: 0.40
Epoch 12:
[Step 4050] Average Loss: 1.864 | Accuracy: 0.42
Epoch 13:
[Step 4050] Average Loss: 1.819 | Accuracy: 0.43
Epoch 14:
[Step 4050] Average Loss: 1.782 | Accuracy: 0.45
Epoch 15:
[Step 4050] Average Loss: 1.749 | Accuracy: 0.46
Epoch 16:
[Step 4050] Average Loss: 1.706 | Accuracy: 0.47
Epoch 17:
[Step 4050] Average Loss: 1.675 | Accuracy: 0.48
Epoch

In [86]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPool2D, Flatten, Dense
from keras.optimizers import SGD

train_images1, test_images1, train_labels1, test_labels1 = train_test_split(np.expand_dims(data,3), 
                                                                            np.expand_dims(labels,-1), test_size=0.1)

train_images1 = (train_images1 / 255) - 0.5
test_images1 = (test_images1 / 255) - 0.5

model = Sequential([
    Conv2D(6, (5,5), activation = "tanh"), #64x64 -> 62x62x8
    MaxPool2D(),      #62x62x8 -> 31x31x8
    Conv2D(16, (5,5), activation = "tanh"),
    MaxPool2D(),
    Flatten(),
    Dense(kinds*kinds, activation="tanh"),
    Dense(kinds, activation="softmax")
])

model.compile(
    optimizer = SGD(0.01),
    loss = "sparse_categorical_crossentropy",
    metrics = ["accuracy"]
)

model.fit(
    train_images1, train_labels1,
    epochs = 6,
    batch_size = 15
)

print(model.summary())

print("Test Loss: %.4f Test Accuracy: %.4f" % tuple(model.evaluate(test_images1, test_labels1)))


Epoch 1/6
270/270 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.0942 - loss: 2.6922
Epoch 2/6
270/270 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.3173 - loss: 2.3118
Epoch 3/6
270/270 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.4827 - loss: 1.7597
Epoch 4/6
270/270 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.5737 - loss: 1.4623
Epoch 5/6
270/270 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6051 - loss: 1.3252
Epoch 6/6
270/270 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6324 - loss: 1.1876


Model: "sequential_36"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_51 (Conv2D)              │ (15, 60, 60, 6)        │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_41 (MaxPooling2D) │ (15, 30, 30, 6)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_52 (Conv2D)              │ (15, 26, 26, 16)       │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_42 (MaxPooling2D) │ (15, 13, 13, 16)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_36 (Flatten)            │ (15, 2704)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_42 (Dense)                │ (15, 225)              │       608,625 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_43 (Dense)                │ (15, 15)               │         3,390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 614,589 (2.34 MB)

 Trainable params: 614,587 (2.34 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

None
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5909 - loss: 1.3386  
Test Loss: 1.2699 Test Accuracy: 0.6178
